Name: Sahil Suresh Ghogare

Roll no.: 23102A0061

Branch / Div:  BE CMPN A

R_Programming_Experiment_3

Installing / loading required packages

In [7]:
install.packages(c("readr", "jsonlite", "readxl", "dplyr", "RSQLite", "DBI", "writexl"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [8]:
library(readr)
library(jsonlite)
library(readxl)
library(dplyr)
library(RSQLite)
library(DBI)

Uploading the dataset

In [9]:
if (file.exists("/usr/lib/python3/dist-packages/google/colab/__init__.py") ||
    Sys.getenv("COLAB_RELEASE_TAG") != "") {
  system("python3 -c \"from google.colab import files; files.upload()\"")
} else {
  cat("If the upload prompt did not appear, upload 'Online Retail.xlsx' manually using the Files panel on the left.\n")
}

In [10]:
list.files()

[1] "Online Retail.xlsx" "sample_data"

Reading in the raw dataset

In [11]:
raw_data <- read_excel("Online Retail.xlsx")
dim(raw_data)

[1] 541909      8

In [12]:
str(raw_data)

tibble [541,909 × 8] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr [1:541909] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ UnitPrice  : num [1:541909] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Country    : chr [1:541909] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [13]:
head(raw_data)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [14]:
transactions_raw <- raw_data %>%
  select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate)

write_csv(transactions_raw, "transactions.csv")
head(transactions_raw)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


In [15]:
products_raw <- raw_data %>%
  select(StockCode, Description, UnitPrice) %>%
  distinct(StockCode, .keep_all = TRUE)

write(toJSON(products_raw, pretty = TRUE), "products.json")
head(products_raw)

StockCode,Description,UnitPrice
<chr>,<chr>,<dbl>
85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
71053,WHITE METAL LANTERN,3.39
84406B,CREAM CUPID HEARTS COAT HANGER,2.75
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
22752,SET 7 BABUSHKA NESTING BOXES,7.65


In [16]:
customers_raw <- raw_data %>%
  select(CustomerID, Country) %>%
  filter(!is.na(CustomerID)) %>%
  distinct(CustomerID, .keep_all = TRUE)

writexl::write_xlsx(customers_raw, "customers.xlsx")
head(customers_raw)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


Task 1: Import and Clean the Data

In [17]:
transactions <- read_csv("transactions.csv", show_col_types = FALSE)
products <- fromJSON("products.json")
customers <- read_excel("customers.xlsx")

In [18]:
str(transactions)

spc_tbl_ [541,909 × 5] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 - attr(*, "spec")=
  .. cols(
  ..   InvoiceNo = col_character(),
  ..   StockCode = col_character(),
  ..   CustomerID = col_double(),
  ..   Quantity = col_double(),
  ..   InvoiceDate = col_datetime(format = "")
  .. )
 - attr(*, "problems")=<pointer: 0x55effe2bccd0> 


In [19]:
str(products)

'data.frame':	4070 obs. of  3 variables:
 $ StockCode  : chr  "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr  "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ UnitPrice  : num  2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...


In [20]:
str(customers)

tibble [4,372 × 2] (S3: tbl_df/tbl/data.frame)
 $ CustomerID: num [1:4372] 17850 13047 12583 13748 15100 ...
 $ Country   : chr [1:4372] "United Kingdom" "United Kingdom" "France" "United Kingdom" ...


In [21]:
colSums(is.na(transactions))

InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0

In [22]:
transactions <- distinct(transactions)
transactions <- transactions %>% filter(!is.na(CustomerID))
transactions <- transactions %>% filter(Quantity > 0)

nrow(transactions)

[1] 392708

In [23]:
products <- products %>% filter(UnitPrice > 0)
products <- products %>% filter(!is.na(Description))

nrow(products)

[1] 3855

Task 2: Integrate the Multiple Data Sources

In [24]:

sales_data <- transactions %>%
  left_join(products, by = "StockCode") %>%
  left_join(customers, by = "CustomerID")

dim(sales_data)

[1] 392708      8

In [25]:
unmatched_products <- sales_data %>% filter(is.na(Description))
unmatched_customers <- sales_data %>% filter(is.na(Country))

cat("Unmatched product rows:", nrow(unmatched_products), "\n")
cat("Unmatched customer rows:", nrow(unmatched_customers), "\n")

Unmatched product rows: 4831 
Unmatched customer rows: 0 


In [26]:
sales_data <- sales_data %>% filter(!is.na(UnitPrice))

sales_data <- sales_data %>% mutate(Revenue = Quantity * UnitPrice)

head(sales_data)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<chr>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom,15.30


Task 3: Sales and Customer Analysis

In [27]:
total_revenue <- sum(sales_data$Revenue, na.rm = TRUE)
total_revenue

[1] 10752840

In [28]:
top_products <- sales_data %>%
  group_by(Description) %>%
  summarise(TotalRevenue = sum(Revenue)) %>%
  arrange(desc(TotalRevenue)) %>%
  head(5)

top_products

Description,TotalRevenue
<chr>,<dbl>
"PAPER CRAFT , LITTLE BIRDIE",168469.60
PARTY BUNTING,142437.56
REGENCY CAKESTAND 3 TIER,135604.80
WHITE HANGING HEART T-LIGHT HOLDER,93745.65
MEDIUM CERAMIC TOP STORAGE JAR,81032.64


In [29]:
top_countries <- sales_data %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue)) %>%
  arrange(desc(TotalRevenue)) %>%
  head(5)

top_countries

Country,TotalRevenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


In [30]:
top_customers <- sales_data %>%
  group_by(CustomerID) %>%
  summarise(TotalPurchase = sum(Revenue)) %>%
  arrange(desc(TotalPurchase)) %>%
  head(5)

top_customers

CustomerID,TotalPurchase
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [31]:
customer_totals <- sales_data %>%
  group_by(CustomerID) %>%
  summarise(TotalPurchase = sum(Revenue))

summary(customer_totals$TotalPurchase)

     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
     1.25    361.00    802.60   2478.18   2000.64 408759.96 

In [32]:
customer_summary <- customer_totals %>%
  mutate(CustomerSegment = case_when(
    TotalPurchase < 500   ~ "Low Value",
    TotalPurchase < 1500  ~ "Medium Value",
    TotalPurchase < 5000  ~ "High Value",
    TRUE                  ~ "Premium"
  ))

table(customer_summary$CustomerSegment)


  High Value    Low Value Medium Value      Premium 
        1049         1516         1425          349 

In [33]:
best_market <- top_countries[1, ]

worst_market <- sales_data %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue)) %>%
  arrange(TotalRevenue) %>%
  head(1)

cat("High-performing market:", best_market$Country, "with revenue", best_market$TotalRevenue, "\n")
cat("Underperforming market:", worst_market$Country, "with revenue", worst_market$TotalRevenue, "\n")

High-performing market: United Kingdom with revenue 8861857 
Underperforming market: Saudi Arabia with revenue 181.44 


Task 4: Store and Retrieve Data Using SQL

In [34]:
con <- dbConnect(RSQLite::SQLite(), "retail_sales.db")
dbWriteTable(con, "retail_sales", sales_data, overwrite = TRUE)

In [35]:
query1 <- dbGetQuery(con, "
  SELECT CustomerID, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5
")

query1

CustomerID,TotalRevenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


In [36]:
query2 <- dbGetQuery(con, "
  SELECT Country, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY TotalRevenue DESC
")

query2

Country,TotalRevenue
<chr>,<dbl>
United Kingdom,8861857.13
Netherlands,363884.48
EIRE,331660.17
Germany,263818.97
France,226975.60
Australia,173918.61
Spain,67426.09
Switzerland,66619.97
Japan,48600.22
